# Table of Contents

1. Pulling the Data
1. Preparing the Data
1. Further Cleaning

# Pulling the Data

All of the data collected from others was collected

In [147]:
# !pip install gspread pandas google-auth
import gspread
from google.oauth2.service_account import Credentials
import os, pandas as pd
from pathlib import Path


In [148]:
os.environ["GOOGLE_SHEETS_CREDENTIALS"] = (
    "/opt/notebooks/psalms_nlp_sp26/private/psalms-blind-scoring-da42a38adf3c.json"
)

os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [149]:
# Current notebook directory
notebook_dir = Path.cwd()

# Build path to the JSON
cred_path = notebook_dir.parent / "private" / "psalms-blind-scoring-da42a38adf3c.json"

print("Credential path exists?", cred_path.exists())

os.getcwd()

Credential path exists? True


'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [150]:
creds = Credentials.from_service_account_file(
    os.environ["GOOGLE_SHEETS_CREDENTIALS"],
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

client = gspread.authorize(creds)

sheet = client.open("results_scored")

In [151]:
# Access second sheet (index 1)
worksheet2 = sheet.get_worksheet(1)

# Get all values
data = worksheet2.get_all_values()

# Convert to DataFrame (first row as header)
df = pd.DataFrame(data[1:], columns=data[0])

# storing the collected data for reference later
df.to_csv("../data/results_scored.csv", index=False, mode="w")


# Preparing the data
I want each row to hold one of the four possible scored for each of the `236 results`. So if rebuilt properly, we should end up with a total of: 
$$236 * 4 = 944\ results$$

I need to start by unpivoting my own score separte from the other scores.

## Numbering the Results 
I also want to be able to reference the order of the results within each search. I collected the top 5 results from each search. There was a bug in my code that took the top 6 results from searches. I am going to just worry about the top 5 results to keep everything fair. 

In [152]:
# temporary dataframe to not break the original 
temp = df.copy()

# aqdding a column to number the indivudal results
temp['numbered_result'] = pd.NA
temp

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,numbered_result
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and j...,9,,,,,,,NaN
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an o...,6,,,,,,,NaN
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning unders...,3,7,p06,,,,,NaN
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose ini...,10,7,p06,,,,,NaN
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a...,7,2,p01,,,,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist ...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, a...",2,0,p01,0,p06,,,NaN
232,Verses where the psalmist ...,TFIDF,7.79,Bible,130,1An ode of ascents by Davi...,1,0,p03,,,,,NaN
233,Verses where the psalmist ...,TFIDF,7.08,Psalter,61,Shall not my soul be subje...,3,10,p06,,,,,NaN
234,Verses where the psalmist ...,TFIDF,5.61,Psalter,35,"The transgressor, that he ...",8,,,,,,,NaN


In [153]:
n = temp.shape[0]

# starting with the number 1 result of a query
num_result = 1

query = temp["Query"].iloc[0]
method = temp["Method"].iloc[0]

for i in range(n):
    # checking if we are in the same group fo data to be numbered
    if query == temp["Query"].iloc[i] and method == temp["Method"].iloc[i]:
        temp["numbered_result"].iloc[i] = num_result
        num_result += 1
        
    # in an new group of results
    else:
        # the current result is the number result for the new set of results
        temp["numbered_result"].iloc[i] = 1
        # reset the number result
        num_result = 2
        # update to the new target query & method
        query = temp["Query"].iloc[i]
        method = temp["Method"].iloc[i]
        
temp.tail(20)


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,numbered_result
216,Verses where the psalmist ...,TFIDF_GLoVe,21.41,Psalter,59,"O God, Thou hast cast us o...",2,8,p01,6,p03,,,3
217,Verses where the psalmist ...,TFIDF_GLoVe,20.54,Bible,61,For the End for Jeduthun a...,8,1,p03,,,,,4
218,Verses where the psalmist ...,TFIDF_GLoVe,20.02,Psalter,103,"Bless the Lord, O my soul....",4,4,p01,0,p06,0,p03,5
219,Verses where the psalmist ...,TFIDF_GLoVe,19.50,Psalter,23,"The earth is the Lord's, a...",4,1,p01,,,,,6
220,Verses where the psalmist ...,BERT,99.88,Bible,32,By David Rejoice greatly i...,1,6,p01,7,p05,2,p06,1
221,Verses where the psalmist ...,BERT,99.87,Bible,8,For the End concerning the...,9,2,p01,0,p03,,,2
222,Verses where the psalmist ...,BERT,99.87,Bible,17,1For the End by the child ...,10,,,,,,,3
223,Verses where the psalmist ...,BERT,99.87,Bible,22,A psalm by David The Lord ...,7,3,p01,,,,,4
224,Verses where the psalmist ...,BERT,99.87,Bible,34,By David OLord judge those...,8,2,p02,,,,,5
225,Verses where the psalmist ...,SBERT,97.02,Psalter,87,O Lord God of my salvation...,3,2,p01,2,p03,,,1


In [154]:
# select first 7 columns + last column
cols_to_keep = list(temp.columns[:7]) + [temp.columns[-1]]
caden = temp[cols_to_keep]

caden.head()


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,numbered_result
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and j...,9,1
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an o...,6,2
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning unders...,3,3
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose ini...,10,4
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a...,7,5


In [155]:
caden['User'] = 'caden'

caden = caden.rename(columns={"CadenScore": "Score"})

caden

/tmp/ipykernel_178/3532660364.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caden['User'] = 'caden'


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,Score,numbered_result,User
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and j...,9,1,caden
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an o...,6,2,caden
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning unders...,3,3,caden
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose ini...,10,4,caden
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a...,7,5,caden
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist ...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, a...",2,2,caden
232,Verses where the psalmist ...,TFIDF,7.79,Bible,130,1An ode of ascents by Davi...,1,3,caden
233,Verses where the psalmist ...,TFIDF,7.08,Psalter,61,Shall not my soul be subje...,3,4,caden
234,Verses where the psalmist ...,TFIDF,5.61,Psalter,35,"The transgressor, that he ...",8,5,caden


In [156]:
caden  = caden[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User', 'Score']]
caden

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,caden,7
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist ...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, a...",caden,2
232,Verses where the psalmist ...,TFIDF,3,7.79,Bible,130,1An ode of ascents by Davi...,caden,1
233,Verses where the psalmist ...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subje...,caden,3
234,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",caden,8


Moving on to prepaering the external scores.

In [157]:
external = temp[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']]

external

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User1,Score1,User2,Score2,User3,Score3
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,,,,,,
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,,,,,,
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,p06,7,,,,
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,p06,7,,,,
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,p01,2,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist ...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, a...",p01,0,p06,0,,
232,Verses where the psalmist ...,TFIDF,3,7.79,Bible,130,1An ode of ascents by Davi...,p03,0,,,,
233,Verses where the psalmist ...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subje...,p06,10,,,,
234,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,,,,,


In [158]:
print(external.columns.tolist())


['Query', 'Method', 'numbered_result', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']


In [159]:
# Unpivot User/Score pairs
df_long = pd.wide_to_long(
    external,
    stubnames=["User", "Score"],  # the base column names
    i=["Query", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", "Verse"],  # columns to keep
    j="Pair",  # new column for the pair number
    sep=""      # number comes directly after the stub name
).reset_index()

# Optional: reorder columns
df_long = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "Pair", "User", "Score"]]



In [168]:
external = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "User", "Score"]]

external.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,,
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,,
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,,
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,,
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,,


In [169]:
caden.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,caden,7


## Combining the Prepared Data back together

In [170]:
scores = pd.concat([caden, external], ignore_index=True)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,caden,7
...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,
940,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,
941,Verses where the psalmist ...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p01,7
942,Verses where the psalmist ...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p06,0


In [171]:
(scores["Score"].notna() & (scores["Score"] != "")).sum()

482

We now have our intended 944 rows of data we can move on to do last few preperations for the analysis. 

# Further Data Cleaning <a href="further_cleaning"></a>

I now need to work on some of the anaiysis of the data and alot of the imediate processiong was done within another notebook so it will be coppied into here. 

## Query Categoization

In [172]:
query_categories = {
    "mercy": "Simple Keyword Queries",
    "prayer":"Simple Keyword Queries",
    "The Lord is my shepherd": "Phrase/Exact Match Queries",
    "Create in me a clean heart":"Phrase/Exact Match Queries",
    "protection from enemies": "Thematic/Semantic Queries",
    "praise in times of suffering": "Thematic/Semantic Queries",
    "How does the psalmist express trust in God while surrounded by fear and uncertainty?":
        "Long/Complex Queries",
    "Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.":
        "Long/Complex Queries",
    "Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.":
        "Orthodox Service Quotes",
    "Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.":
        "Orthodox Service Quotes",
    "For the Peace of the world": "Orthodox Service Quotes"

}

In [173]:
scores["Query Category"] = scores["Query"].map(QUERY_TO_CATEGORY)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score,Query Category
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,caden,9,Phrase/Exact Match Queries
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,caden,6,Phrase/Exact Match Queries
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,caden,3,Phrase/Exact Match Queries
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,caden,10,Phrase/Exact Match Queries
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,caden,7,Phrase/Exact Match Queries
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,,Long/Complex Queries
940,Verses where the psalmist ...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,,Long/Complex Queries
941,Verses where the psalmist ...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p01,7,Long/Complex Queries
942,Verses where the psalmist ...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p06,0,Long/Complex Queries


In [174]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", 
                  "Verse", "User", "Score" ]]

scores

,Query,Query Category,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and j...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an o...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning unders...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose ini...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a...,caden,7
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,
940,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5,5.61,Psalter,35,"The transgressor, that he ...",,
941,Verses where the psalmist ...,Long/Complex Queries,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p01,7
942,Verses where the psalmist ...,Long/Complex Queries,TFIDF,6,5.46,Psalter,131,"Lord, remember David and a...",p06,0


In [175]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method","Similarity Score (%)", "numbered_result",
                  "Text", "Psalm Num", "Verse", "User", "Score" ]]

scores

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and j...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an o...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning unders...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose ini...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a...,caden,7
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he ...",,
940,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he ...",,
941,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.46,6,Psalter,131,"Lord, remember David and a...",p01,7
942,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.46,6,Psalter,131,"Lord, remember David and a...",p06,0


In [176]:
# filtering to only be studying the top 5 results from each query
scores = scores[scores['numbered_result'] != 6]


pd.set_option("display.max_rows", 50)

scores[scores['Method'] == 'TFIDF']

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
16,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,15.05,1,Psalter,50,"Have mercy upon me, O God,...",caden,10
17,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,14.21,2,Psalter,54,"Give ear to my prayer, O G...",caden,9
18,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,14.20,3,Psalter,23,"The earth is the Lord's, a...",caden,6
19,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,12.53,4,Bible,50,For the End a psalm by Dav...,caden,8
20,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,12.48,5,Psalter,40,Blessed is he that conside...,caden,4
...,...,...,...,...,...,...,...,...,...,...
936,Verses where the psalmist ...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subje...,,
937,Verses where the psalmist ...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subje...,,
938,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he ...",,
939,Verses where the psalmist ...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he ...",,


---

# Analysis